### Imports

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import numpy as np 
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import mean_squared_error,r2_score, mean_absolute_error,accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import optuna
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBRegressor

### Model Training

Splits the dataset into training (80%) and testing (20%) sets using a fixed random seed for reproducibility. The variables X and y are expected to exist from prior preprocessing steps.

In [20]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [21]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.fit_transform(X_test)

Saves the fitted scaler to a file (scaler.pkl) so it can be reused later (e.g., for transforming new data during deployment). Even though the scaler was incorrectly refitted, it is still saved.

In [42]:
import joblib
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

Usesed the LazyRegressor library to quickly compare multiple regression models (about 40) without any hyperparameter tuning. It fits each model on the training data and evaluates on the test set, returning a DataFrame with metrics like R², RMSE, and training time. This gives a quick overview of which models perform best before deeper tuning.
Note: The results show that HistGradientBoostingRegressor and LGBMRegressor are among the top performers.


In [41]:
from lazypredict.Supervised import LazyRegressor

reg = LazyRegressor(verbose=0, ignore_warnings=True, predictions=False)

models_reg, predictions_reg = reg.fit(
    X_train, X_test, y_train, y_test
)

models_reg

  0%|          | 0/42 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001503 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1084
[LightGBM] [Info] Number of data points in the train set: 36474, number of used features: 9
[LightGBM] [Info] Start training from score 26.301612


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
HistGradientBoostingRegressor,0.29,0.29,7.90,1.33
MLPRegressor,0.28,0.28,7.92,94.31
GradientBoostingRegressor,0.28,0.28,7.93,7.45
LGBMRegressor,0.27,0.28,7.97,0.87
NuSVR,0.27,0.27,8.01,149.96
SVR,0.26,0.26,8.03,157.64
AdaBoostRegressor,0.25,0.25,8.11,1.58
PoissonRegressor,0.19,0.20,8.40,0.06
LassoCV,0.19,0.20,8.40,0.53


# Optuna (HistGradientBoostingRegressor)

Defines the objective function for Optuna to optimise the hyperparameters of HistGradientBoostingRegressor.

- suggest_float, suggest_int define the search space for each parameter.

- The model is built with those parameters, and 5‑fold cross‑validation is used to compute the negative RMSE (since Optuna maximises, we return the mean negative RMSE).

- The trial.report() and pruning logic allow Optuna to stop unpromising trials early.



In [25]:
def objective(trial):
    
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),  # log scale better explores small values
        "max_iter": trial.suggest_int("max_iter", 100, 500, step=50),               # step reduces redundant trials
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 50),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 1.0, log=True),  # log avoids over-sampling near 0
        "max_bins": trial.suggest_int("max_bins", 128, 255),                        # extra lever often worth tuning
    }
    
    model = HistGradientBoostingRegressor(
        **params,
        early_stopping=True,       # stops adding trees when val score plateaus
        validation_fraction=0.1,   # held-out slice used by early stopping
        n_iter_no_change=10,       # patience before stopping
        random_state=42,
    )
    
    cv = KFold(n_splits=5, shuffle=True, random_state=42)  # shuffling reduces split bias
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",  # RMSE is more interpretable than MSE
        n_jobs=-1,
        error_score="raise",       # surfaces silent failures instead of hiding them
    )
    
    # Report intermediate value for Optuna pruning
    trial.report(scores.mean(), step=0)
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()
    
    return scores.mean()

Trains a final HistGradientBoostingRegressor on the full training set using the best hyperparameters discovered by Optuna. The fitted model is stored.

In [26]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5),
)
study.optimize(objective, n_trials=100, timeout=600)

[I 2026-02-23 12:47:46,691] A new study created in memory with name: no-name-91e10551-0abd-40e2-a1dd-b73b99d7485e
[I 2026-02-23 12:47:57,417] Trial 0 finished with value: -7.440663273829264 and parameters: {'learning_rate': 0.030710573677773714, 'max_iter': 500, 'max_depth': 12, 'min_samples_leaf': 34, 'l2_regularization': 0.00042079886696066364, 'max_bins': 147}. Best is trial 0 with value: -7.440663273829264.
[I 2026-02-23 12:48:07,317] Trial 1 finished with value: -7.534964593387983 and parameters: {'learning_rate': 0.011900590783184251, 'max_iter': 450, 'max_depth': 10, 'min_samples_leaf': 39, 'l2_regularization': 0.00012087541473056971, 'max_bins': 252}. Best is trial 0 with value: -7.440663273829264.
[I 2026-02-23 12:48:13,360] Trial 2 finished with value: -7.434884652938817 and parameters: {'learning_rate': 0.12106896936002161, 'max_iter': 150, 'max_depth': 5, 'min_samples_leaf': 17, 'l2_regularization': 0.0016480446427978971, 'max_bins': 195}. Best is trial 2 with value: -7.434

In [28]:
best_model = HistGradientBoostingRegressor(**study.best_params)
best_model.fit(X_train, y_train)

,loss,'squared_error'
,quantile,None
,learning_rate,0.17470766977719882
,max_iter,400
,max_leaf_nodes,31
,max_depth,4
,min_samples_leaf,41
,l2_regularization,0.594800822345122
,max_features,1.0
,max_bins,220
,categorical_features,'from_dtype'


Evaluates the best HistGradientBoosting model on the test set, computing R², Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE). The results are printed.

In [29]:
# Predictions
y_pred = best_model.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
acc=best_model.score(X_test,y_test)

print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"accuracy: {acc:.2f}")

R² Score: 0.2912
MAE: 6.23
RMSE: 7.88
accuracy: 0.29


# Optuna(Random Forest)

Defines the objective function for Random Forest. The hyperparameter space includes the number of trees, depth, leaf size, split criteria, feature fraction, and bootstrap sample size. The evaluation uses 5‑fold cross‑validation with negative RMSE.

In [57]:
def objective(trial):
    
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.8]),
        "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),  # controls row subsampling (bagging)
        "bootstrap": True,
        "random_state": 42,
        "n_jobs": -1,
    }
    
    model = RandomForestRegressor(**params)
    
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        error_score="raise",
    )
    
    trial.report(scores.mean(), step=0)
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()
    
    return scores.mean()


Runs the Optuna optimisation for Random Forest (100 trials). After completion, the best parameters and the corresponding RMSE (converted from negative) are printed.

In [58]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5),
)
study.optimize(objective, n_trials=100, timeout=600)

print("Best params:", study.best_params)
print("Best RMSE:", -study.best_value)

[I 2026-02-21 13:48:44,459] A new study created in memory with name: no-name-638663d2-010e-4c7f-b8d7-a61890445262
[I 2026-02-21 13:49:00,891] Trial 0 finished with value: -7.602296059244037 and parameters: {'n_estimators': 450, 'max_depth': 20, 'min_samples_leaf': 37, 'min_samples_split': 13, 'max_features': 0.5, 'max_samples': 0.8540362888980227}. Best is trial 0 with value: -7.602296059244037.
[I 2026-02-21 13:49:03,969] Trial 1 finished with value: -7.631566163017025 and parameters: {'n_estimators': 100, 'max_depth': 20, 'min_samples_leaf': 42, 'min_samples_split': 6, 'max_features': 0.5, 'max_samples': 0.645614570099021}. Best is trial 0 with value: -7.602296059244037.
[I 2026-02-21 13:49:09,446] Trial 2 finished with value: -7.902976774938358 and parameters: {'n_estimators': 650, 'max_depth': 5, 'min_samples_leaf': 15, 'min_samples_split': 8, 'max_features': 'log2', 'max_samples': 0.5232252063599989}. Best is trial 0 with value: -7.602296059244037.
[I 2026-02-21 13:49:16,540] Tria

Best params: {'n_estimators': 900, 'max_depth': 18, 'min_samples_leaf': 19, 'min_samples_split': 6, 'max_features': 0.8, 'max_samples': 0.9249713295003107}
Best RMSE: 7.5662586836358345


Trains the final Random Forest model using the optimal hyperparameters on the full training set.

In [60]:
best_model_random_forest = RandomForestRegressor(**study.best_params)
best_model_random_forest.fit(X_train, y_train)

,n_estimators,900
,criterion,'squared_error'
,max_depth,18
,min_samples_split,6
,min_samples_leaf,19
,min_weight_fraction_leaf,0.0
,max_features,0.8
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Evaluates the tuned Random Forest model on the test set, printing R², MAE, and RMSE.

In [62]:
# Predictions
y_pred = best_model_random_forest.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

R² Score: 0.2759
MAE: 6.34
RMSE: 7.97


# Optuna XG boost

Defines the objective function for XGBoost. The hyperparameter space includes many of XGBoost’s key parameters: number of trees, learning rate, tree depth, regularisation terms, subsampling, and column sampling. The tree_method='hist' speeds up training. Again, 5‑fold CV with negative RMSE is used.

In [78]:
def objective(trial):
    
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),   # like min_samples_leaf
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),            # row subsampling per tree
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),  # feature subsampling per tree
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1.0),# feature subsampling per level
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),                   # min loss reduction to split
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),   # L1 regularization
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True), # L2 regularization
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),       # helps with imbalanced targets
        "random_state": 42,
        "tree_method": "hist",   # faster training, equivalent to HistGradientBoosting
        "device": "cpu",         # change to "cuda" if you have a GPU
        "n_jobs": -1,
    }
    
    model = XGBRegressor(**params)
    
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        error_score="raise",
    )
    
    trial.report(scores.mean(), step=0)
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()
    
    return scores.mean()


Runs the XGBoost optimisation (100 trials). After completion, it prints the best hyperparameters and the corresponding RMSE.

In [79]:


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5),
)

study.optimize(objective, n_trials=100, timeout=600)

print("Best params:", study.best_params)
print("Best RMSE: ", -study.best_value)

[I 2026-02-21 14:08:39,970] A new study created in memory with name: no-name-7ab7a41f-20c0-48e3-a5cf-b8064b4ce439
[I 2026-02-21 14:08:42,312] Trial 0 finished with value: -7.824654674530029 and parameters: {'n_estimators': 450, 'learning_rate': 0.2536999076681772, 'max_depth': 10, 'min_child_weight': 6, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.40919616423534183, 'colsample_bylevel': 0.3406585285177396, 'gamma': 4.330880728874676, 'reg_alpha': 0.10129197956845731, 'reg_lambda': 0.3470266988650412, 'max_delta_step': 0}. Best is trial 0 with value: -7.824654674530029.
[I 2026-02-21 14:08:46,051] Trial 1 finished with value: -7.524778842926025 and parameters: {'n_estimators': 1000, 'learning_rate': 0.16967533607196555, 'max_depth': 5, 'min_child_weight': 2, 'subsample': 0.5917022549267169, 'colsample_bytree': 0.5129695700716763, 'colsample_bylevel': 0.6673295021425665, 'gamma': 2.1597250932105787, 'reg_alpha': 0.0028585493941961923, 'reg_lambda': 0.11462107403425033, 'max_del

Best params: {'n_estimators': 1000, 'learning_rate': 0.03984086353065521, 'max_depth': 4, 'min_child_weight': 7, 'subsample': 0.8766167417779783, 'colsample_bytree': 0.8289141651973928, 'colsample_bylevel': 0.860652392387826, 'gamma': 1.4706590440584235, 'reg_alpha': 0.11509515062921359, 'reg_lambda': 0.00840593157875983, 'max_delta_step': 9}
Best RMSE:  7.386575508117676


Trains the final XGBoost model with the best found parameters on the full training set.

In [83]:
best_model_xgboost = XGBRegressor(**study.best_params)
best_model_xgboost.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,0.860652392387826
,colsample_bynode,None
,colsample_bytree,0.8289141651973928
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


Evaluates the tuned XGBoost model on the test set and prints the metrics.

In [84]:
# Predictions
y_pred = best_model_xgboost.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score: {r2:.4f}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

R² Score: 0.1303
MAE: 6.83
RMSE: 8.73
